# C2：特征有用吗，动作能改变它吗

低 feature loss 可能只是找到了容易预测的常量。本实验增加线性探针、反事实动作和短期动作选择。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import torch
from hwm.data import make_pixelworld_dataset
from hwm.jepa import (
    TinyVideoJEPA, apply_linear_probe, feature_spread,
    fit_linear_probe_weights, jepa_batch_from_episodes,
)
torch.manual_seed(2)


## 1. 训练 Action-JEPA

最后一个动作决定历史倒数第二帧怎样变成最后一帧。与被动 JEPA 相比，Predictor 多读一个 action embedding。

In [ ]:
train_episodes = make_pixelworld_dataset(10, 7, seed=2)
test_episodes = make_pixelworld_dataset(4, 7, seed=31)
video, actions, positions = jepa_batch_from_episodes(
    train_episodes, history_length=3
)
test_video, test_actions, test_positions = jepa_batch_from_episodes(
    test_episodes, history_length=3
)
model = TinyVideoJEPA(feature_size=16)
parameters = list(model.online_encoder.parameters()) + list(model.predictor.parameters()) + list(model.action_embedding.parameters())
optimizer = torch.optim.Adam(parameters, lr=3e-3)
losses = []
for _ in range(40):
    optimizer.zero_grad(); loss, prediction, target, features = model.loss(video, actions); loss.backward(); optimizer.step(); model.update_target(0.99); losses.append(float(loss.detach()))
print('loss:', round(losses[0], 3), '→', round(losses[-1], 3), 'spread:', round(float(feature_spread(features).detach()), 3))
assert losses[-1] < losses[0]


## 2. 线性探针：能否读出位置

冻结表示，只用一个线性映射读方块中心。如果连这个简单属性都读不出，特征很难支持后续空间任务。

In [ ]:
with torch.no_grad():
    _, train_target, _ = model(video, actions)
    _, test_target, _ = model(test_video, test_actions)
# 平均 patch 会抹掉位置；flatten 保留每个 patch 在网格中的槽位。
train_features = train_target.flatten(1)
test_features = test_target.flatten(1)
probe_weights = fit_linear_probe_weights(train_features, positions)
test_prediction = apply_linear_probe(test_features, probe_weights)
probe_mse = torch.nn.functional.mse_loss(test_prediction, test_positions)
constant = positions.mean(0).expand_as(test_positions)
constant_mse = torch.nn.functional.mse_loss(constant, test_positions)
print('held-out probe/base MSE:', round(float(probe_mse), 4),
      round(float(constant_mse), 4))
assert probe_mse < constant_mse


## 3. 固定历史，只替换动作

被动视频表示可以保存运动，却不能证明模型懂得控制。Action-JEPA 至少应让候选未来随动作变化。

In [ ]:
same_history = video[:1].expand(5, -1, -1, -1, -1)
all_actions = torch.arange(5)
with torch.no_grad(): predictions, _, _ = model(same_history, all_actions)
differences = [(predictions[0] - predictions[i]).square().mean().item() for i in range(1, 5)]
print('换动作后的 feature MSE:', [round(x, 5) for x in differences])
assert max(differences) > 0


## 4. 一个最小动作选择接口

同一个 held-out 位置探针把五个候选 feature 变回位置，再选择更接近绿色目标的动作。若 probe 在新 episode 上不可靠，这个动作选择也没有可信依据。

In [ ]:
predicted_positions = apply_linear_probe(
    predictions.flatten(1), probe_weights
)
goal = torch.tensor([12 / 15, 12 / 15])
distances = torch.linalg.vector_norm(predicted_positions - goal, dim=-1)
chosen_action = int(distances.argmin())
print('候选位置:', predicted_positions)
print('到目标距离:', [round(float(x), 4) for x in distances])
print('选择动作:', chosen_action)
assert 0 <= chosen_action < 5


## 小结

线性探针回答‘特征里有什么’，反事实回答‘动作是否进入动态’，短期选择回答‘表示能否接到行动’。它们都比单独汇报 feature loss 更接近世界模型的用途。